# 保存済み FunctionGemma を端末用に変換
学習は繰り返しません。Google Drive の `tsunami-functiongemma-output/model` を `dynamic_int8` 量子化し、`.litertlm` を同じフォルダ内の `litertlm` に保存します。変換後の正答率と端末速度は別途確認が必要です。
Google の [Mobile Actions ノートブック](https://ai.google.dev/gemma/docs/mobile-actions) の変換手順を、今回の保存先に合わせたものです。新しい Colab ランタイムで上から実行してください。

In [ ]:
%pip uninstall -y tensorflow
%pip install ai-edge-torch-nightly --force-reinstall
%pip install ai-edge-litert-nightly

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/tsunami-functiongemma-output')
saved_model = drive_root / 'model'
for name in ('model.safetensors', 'config.json', 'tokenizer.model'):
    assert (saved_model / name).is_file(), f'保存済みモデルに {name} がありません: {saved_model}'
local_model = Path('/content/tsunami-functiongemma-model')
shutil.copytree(saved_model, local_model, dirs_exist_ok=True)
print('変換元:', local_model)
print('重みの大きさ:', round((local_model / 'model.safetensors').stat().st_size / 1_000_000, 1), 'MB')

In [ ]:
from ai_edge_torch.generative.examples.gemma3 import gemma3
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig
from ai_edge_torch.generative.layers import kv_cache
from importlib.metadata import version

out_dir = Path('/content/tsunami-litertlm')
out_dir.mkdir(parents=True, exist_ok=True)
metadata = out_dir / 'base_llm_metadata.textproto'
metadata.write_text('''start_token: {
    token_ids: {
        ids: [ 2 ]
    }
}
stop_tokens: {
    token_str: "<end_of_turn>"
}
stop_tokens: {
    token_str: "<start_function_response>"
}
llm_model_type: {
    function_gemma: {}
}
''', encoding='utf-8')
print('ai-edge-torch-nightly:', version('ai-edge-torch-nightly'))
print('ai-edge-litert-nightly:', version('ai-edge-litert-nightly'))
model = gemma3.build_model_270m(str(local_model))
export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True
converter.convert_to_litert(
    model, output_path=str(out_dir), output_name_prefix='tsunami-functiongemma',
    prefill_seq_len=512, kv_cache_max_len=1024, quantize='dynamic_int8',
    export_config=export_config, tokenizer_model_path=str(local_model / 'tokenizer.model'),
    base_llm_metadata_path=str(metadata), output_format='litertlm',
)
converted = list(out_dir.glob('*.litertlm'))
assert len(converted) == 1, f'変換結果が1個ではありません: {converted}'
size_mb = converted[0].stat().st_size / 1_000_000
print('変換結果:', converted[0], f'{size_mb:.1f} MB')
if size_mb > 300:
    print('注意: 端末用の目標 300 MB を超えています')
drive_output = drive_root / 'litertlm'
drive_output.mkdir(exist_ok=True)
saved = Path(shutil.copy2(converted[0], drive_output / converted[0].name))
assert saved.stat().st_size == converted[0].stat().st_size, 'Drive へのコピーが不完全です'
print('Drive保存先:', saved)